### Cốt lõi của thuật toán (UCS)

In [1]:
from collections import deque
import heapq

class Node:
    def __init__(self, state, parent=None, action=None, cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost

    def __lt__(self, other):
        return self.cost < other.cost

    def get_path(self):
        path = []
        cur_node = self
        while cur_node:
            path.append((cur_node.state, cur_node.action))
            cur_node = cur_node.parent
        return path[::-1]

class Problem:
    def __init__(self, start_state, target_state, get_moves, apply_move, step_cost=None):
        self.start_state = start_state
        self.target_state = target_state
        self.get_moves = get_moves
        self.apply_move = apply_move
        self.step_cost = step_cost

def is_goal(problem, cur_state):
    if callable(problem.target_state):
        return problem.target_state(cur_state)
    return cur_state == problem.target_state

def UCS(problem: Problem):
    initial_node = Node(problem.start_state)

    frontier = []
    heapq.heappush(frontier, initial_node)
    explored = {initial_node.state: 0}

    while frontier:
        cur_node = heapq.heappop(frontier)

        if explored[cur_node.state] < cur_node.cost:
            continue

        if is_goal(problem, cur_node.state):
            return cur_node.get_path()

        for action in problem.get_moves(cur_node.state):
            next_state = problem.apply_move(cur_node.state, action)
            new_cost = cur_node.cost + problem.step_cost(cur_node.state, action)

            if next_state not in explored or explored[next_state] > new_cost:
                child = Node(next_state, cur_node, action=action, cost=new_cost)
                explored[next_state] = new_cost
                heapq.heappush(frontier, child)

    return None

### Bài toán 8-Puzzle

In [2]:
def find_blank(cur_board):
    for row_idx in range(3):
        for col_idx in range(3):
            if cur_board[row_idx][col_idx] == 0:
                return row_idx, col_idx

def actions_8puzzle(cur_board):
    blank_r, blank_c = find_blank(cur_board)
    valid_moves = []

    if blank_r > 0: valid_moves.append("Lên")
    if blank_r < 2: valid_moves.append("Xuống")
    if blank_c > 0: valid_moves.append("Trái")
    if blank_c < 2: valid_moves.append("Phải")

    return valid_moves

def result_8puzzle(cur_board, move):
    blank_r, blank_c = find_blank(cur_board)

    new_r, new_c = blank_r, blank_c
    if move == "Lên":
        new_r -= 1
    elif move == "Xuống":
        new_r += 1
    elif move == "Trái":
        new_c -= 1
    elif move == "Phải":
        new_c += 1

    grid = [list(cur_board[0]), list(cur_board[1]), list(cur_board[2])]
    grid[blank_r][blank_c] = grid[new_r][new_c]
    grid[new_r][new_c] = 0

    return (tuple(grid[0]), tuple(grid[1]), tuple(grid[2]))

def cost_8puzzle(cur_board, move):
    if move in ("Lên", "Xuống"):
        return 1
    return 2

def print_state(board):
    for r in range(3):
        print(f"[{board[r][0]} {board[r][1]} {board[r][2]}]")
    print()

if __name__ == "__main__":
    start_board = ((1, 2, 3), (4, 0, 6), (7, 5, 8))
    target_board = ((1, 2, 3), (4, 5, 6), (7, 8, 0))
    puzzle = Problem(start_board, target_board, actions_8puzzle, result_8puzzle, cost_8puzzle)

    steps = UCS(puzzle)
    if steps is None:
        print("Không tìm thấy lời giải")
    else:
        print("SOLUTION")
        for step_idx in range(len(steps)):
            board, action = steps[step_idx]
            if step_idx == len(steps) - 1:
                print(f"Bước {step_idx} ({action}) (GOAL):")
            else:
                print(f"Bước {step_idx} ({action}):")
            print_state(board)

SOLUTION (Tổng chi phí: 3)
Bước 0, g = 0:
[1 2 3]
[4 0 6]
[7 5 8]

Bước 1, g = 1:
[1 2 3]
[4 5 6]
[7 0 8]

Bước 2 (GOAL), g = 3:
[1 2 3]
[4 5 6]
[7 8 0]


### Bài toán Vacuum Robot

In [3]:
def actions_vacuum(cur_state):
    return ["Hút", "Trái", "Phải"]

def result_vacuum(cur_state, move):
    pos, room_a, room_b = cur_state

    if move == "Trái":
        return ("A", room_a, room_b)
    elif move == "Phải":
        return ("B", room_a, room_b)
    elif move == "Hút":
        if pos == "A":
            return (pos, "Sạch", room_b)
        else:
            return (pos, room_a, "Sạch")

def cost_vacuum(cur_state, move):
    if move == "Hút":
        return 1
    return 2

def goal_vacuum(cur_state):
    return cur_state[1] == "Sạch" and cur_state[2] == "Sạch"

def print_vacuum_state(cur_state):
    pos, room_a, room_b = cur_state
    print(f"Robot ở phòng {pos} | Phòng A: {room_a} | Phòng B: {room_b}")

if __name__ == "__main__":
    start_vacuum = ("A", "Bẩn", "Bẩn")
    vacuum_puzzle = Problem(start_vacuum, goal_vacuum, actions_vacuum, result_vacuum, cost_vacuum)

    steps = UCS(vacuum_puzzle)
    if steps is None:
        print("Không tìm thấy lời giải")
    else:
        print("SOLUTION")
        for step_idx in range(len(steps)):
            state, action = steps[step_idx]
            if step_idx == len(steps) - 1:
                print(f"Bước {step_idx} ({action}) (GOAL):")
            else:
                print(f"Bước {step_idx} ({action}):")
            print_vacuum_state(state)

SOLUTION (Tổng chi phí: 4)
Bước 0, g = 0:
Robot ở phòng A | Phòng A: Bẩn | Phòng B: Bẩn
Bước 1, g = 1:
Robot ở phòng A | Phòng A: Sạch | Phòng B: Bẩn
Bước 2, g = 3:
Robot ở phòng B | Phòng A: Sạch | Phòng B: Bẩn
Bước 3 (GOAL), g = 4:
Robot ở phòng B | Phòng A: Sạch | Phòng B: Sạch
